# 03. Мощность и устойчивость: налипание/намокание, град, перегрев (угрозы №7, 8, 9, 1, 15)

**Данные только из ТЗ:** БВС-5 (`power_consumption`, `wind_gust`, `wind_speed`, `temperature`, `stabilization_status`; 1 Гц), БВС-1 (`speed`, `pitch`, `roll`; ≈ 8 Гц), БВС-7 (`system_health.cpu_temp`, `component_status`, `error_codes`; 0,14 Гц + по событию), НСУ-3 (`precipitation_prob`, `turbulence_index`).

## Физические следы угроз в этих полях
| Угроза | След в потоках | Детектор |
|---|---|---|
| 8 Снег (налипание), 7 Дождь (намокание) | рост потребляемой мощности при тех же скорости и ветре («рост веса» из таблицы угроз) | остаток `P / P_ожид − 1`, сглаженная прибавка δ̂ |
| 9 Град | серия скачков `pitch/roll`, события ошибки IMU (`0x2101`), `precipitation_prob` и `turbulence_index` | признаки по окну + логрегрессия + подтверждение |
| 1, 15 Пыль (перегрев, абразив) | рост `cpu_temp` относительно ожидаемого при данной T, статус ESC, код `0x3105` | остаток температуры + CUSUM |

Время: БВС-1 идёт ≈ 8 Гц, поэтому обработка его сообщения - горячий путь (должна быть единицы мкс); решения принимаются на тике БВС-5 (1 Гц).

> **Честность оценки.** Связи погоды с полями заданы генератором (ноутбук 00): прибавка мощности от снега (≈ 0,08 % в секунду при максимальной интенсивности, до 25 %), удары града (импульсы 3-8° и коды IMU), нагрев процессора пылью (до +14 °C) - **допущения**, а не измерения. Поэтому результаты показывают работоспособность метода и компромиссы; для града добавлены стресс-тесты (слабый град, турбулентность без града, потери сообщений).

In [ ]:
import sys, pathlib, warnings, time, math
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from src.eval_power import (KINDS3, build_cache3, eval_events, build_hail_rows)
from src.eval_bvs5 import segments
from src.detectors_power import (PowerResidualDetector, HailDetector, HailFeatures, DustThermalDetector, fit_power_model, fit_cpu_model, HAIL_FEATS)
from src.streams import DT
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
t0 = time.time()
train = build_cache3(KINDS3, range(300, 312))
theta, sig_p = fit_power_model(train); cpu_model = fit_cpu_model(train)
Xtr, ytr, _ = build_hail_rows(train); del train
tune = build_cache3(KINDS3, range(200, 208)); test = build_cache3(KINDS3, range(400, 408))
print("модель мощности θ =", np.round(theta, 3), "| σ остатка =", round(sig_p, 4), "| модель cpu:", np.round(cpu_model, 3), "| время", round(time.time() - t0), "с")
print("обучающих строк для града:", Xtr.shape, "| доля града:", round(float(ytr.mean()), 4))

## A. Налипание снега и намокание: остаток мощности
**Модель.** `P_ожид = θ₀ + θ₁·v² + θ₂·g_s`, где `v` - скорость из БВС-1, `g_s` - порыв БВС-5, **сглаженный EWMA (10 с)**: измеренные порывы шумные, и без сглаживания порывы при граде и пыли давали ложные тревоги (оцененный коэффициент занижается из-за шума регрессора). Коэффициенты - ridge-регрессия на безопасных полётах (без осадков, пыли). Остаток `r = P/P_ожид − 1`; прибавка `δ̂ = EWMA(r, τ)`. Алерт: `δ̂ ≥ порог` n подряд (Gate), прогрев 25 с (переходный процесс набора высоты).

Истина - скрытая прибавка нагрузки из генератора `δ(t)`; событие - интервал `δ > 2,5 %`. Корректный алерт - в окне [onset; конец + 60 с].

In [ ]:
tgt = lambda c_: c_["S"]["delta"] > 0.025
rows = []
for tau in (30, 60, 120):
    for on in (0.015, 0.02, 0.03):
        r = eval_events(lambda: PowerResidualDetector(theta, tau, on=on, off=on / 2), tune, tgt, "load_increase"); rows.append(dict(tau=tau, порог=on, **r))
g = pd.DataFrame(rows); display(g.round(3).set_index(["tau", "порог"]))
okp = g[(g.ложных_в_час <= 1.0) & (g.recall >= 0.9)].sort_values("задержка_медиана_с")
TAU, ON = (float(okp.iloc[0].tau), float(okp.iloc[0].порог)) if len(okp) else (60.0, 0.02); print("выбрано: tau =", TAU, "с, порог =", ON, "(критерий: ложных ≤ 1/ч, recall ≥ 0,9, минимум задержки)")
mk = lambda: PowerResidualDetector(theta, TAU, on=ON, off=ON / 2)
rows = [dict(детектор="EWMA + Gate (выбранный)", **eval_events(mk, test, tgt, "load_increase")),
        dict(детектор="CUSUM (k=0,02; h=0,3)", **eval_events(lambda: PowerResidualDetector(theta, 60, mode="cusum", k=0.02, h=0.3), test, tgt, "load_increase"))]
display(pd.DataFrame(rows).set_index("детектор").round(2))
fa_k = {k: eval_events(mk, [c_ for c_ in test if c_["kind"] == k], tgt, "load_increase")["ложных_в_час"] for k in KINDS3}
print("ложных в час по сценариям:", {k: round(v, 2) for k, v in fa_k.items() if v > 0})
print("события по сценариям:", {k: sum(len(segments(tgt(c_))) for c_ in test if c_["kind"] == k) for k in KINDS3 if any(len(segments(tgt(c_))) for c_ in test if c_["kind"] == k)})

In [ ]:
def trace(c_):
    d = mk(); d.reset(); ts, dh, r_, al = [], [], [], []
    for m in c_["msgs"]:
        if m["stream"] in d.streams:
            a = d.on_message(m)
            if m["stream"] == "BVS5": ts.append(m["t_gen"]); dh.append(d.delta_hat)
            al += [m["t_gen"] for _ in a]
    return np.array(ts), np.array(dh), al
cs = next(c_ for c_ in test if c_["kind"] == "snow" and c_["S"]["delta"].max() > 0.08)
ts, dh, al = trace(cs)
fig, ax = plt.subplots(figsize=(9, 3.8)); ax.plot(cs["S"]["t"], cs["S"]["delta"], label="истинная прибавка δ"); ax.plot(ts, dh, label="оценка δ̂"); ax.axhline(ON, c="k", ls=":", label="порог")
ax.vlines(al, 0, 0.02, color="r", lw=2, label="алерт"); ax.legend(fontsize=8); ax.set_xlabel("t, с"); ax.set_ylabel("прибавка мощности"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
# качество оценки уровня
errs = []
for c_ in test:
    if c_["kind"] != "snow": continue
    ts, dh, _ = trace(c_); true = np.interp(ts, c_["S"]["t"], c_["S"]["delta"]); m_ = true > 0.02; errs += list(np.abs(dh[m_] - true[m_]))
print("снег: средняя |ошибка оценки δ̂| в периоды δ > 2 % =", round(float(np.mean(errs)), 4), f"({len(errs)} точек)")

In [ ]:
# граница обнаружения: интенсивность снега
def set_snow(I):
    def f(cfg):
        for e in cfg.events:
            if e.kind == "snow": e.I = I
    return f
rows = []
for I in (0.15, 0.3, 0.6, 1.0):
    ch = build_cache3(["snow"], range(700, 706), modify=set_snow(I))
    r = eval_events(mk, ch, tgt, "load_increase"); mx = np.mean([c_["S"]["delta"].max() for c_ in ch]); rows.append(dict(интенсивность=I, макс_прибавка=mx, **r))
sn = pd.DataFrame(rows).set_index("интенсивность").round(3); display(sn)

## B. Град: скачки attitude + ошибки IMU + прогноз
**Признаки на тике БВС-5 (окна 5 и 15 с):** число скачков `pitch/roll` (отсчёт БВС-1 выше 4 робастных σ, где σ оценивается по медиане |x| за 80 отсчётов, и не менее 2,5°), число событий `0x2101` (БВС-7), `precipitation_prob` и `turbulence_index` (НСУ-3), `gust − wind`, число тиков `stabilization_status ≠ ok` за 5 с. Модель - логрегрессия (чистый numpy на борту), затем Gate (n_on = 2, n_off = 5, пауза 30 с).

In [ ]:
sets = {"только скачки attitude": [0, 1], "только ошибки IMU (БВС-7)": [2, 3], "только прогноз НСУ-3": [4, 5], "attitude + IMU": [0, 1, 2, 3], "attitude + IMU + прогноз": [0, 1, 2, 3, 4, 5], "все признаки": list(range(8))}
Xte, yte, mte = build_hail_rows(test); rows = []
for name, cols in sets.items():
    sc = StandardScaler().fit(Xtr[:, cols]); lr = LogisticRegression(max_iter=500, class_weight="balanced").fit(sc.transform(Xtr[:, cols]), ytr)
    rows.append(dict(набор=name, AUROC=roc_auc_score(yte, lr.predict_proba(sc.transform(Xte[:, cols]))[:, 1])))
display(pd.DataFrame(rows).set_index("набор").round(3))
cols = list(range(8)); sc = StandardScaler().fit(Xtr[:, cols]); lr = LogisticRegression(max_iter=500, class_weight="balanced").fit(sc.transform(Xtr[:, cols]), ytr)
mkh = lambda cols_=None, scx=None, lrx=None: HailDetector((lrx or lr).coef_[0], (lrx or lr).intercept_[0], (scx or sc).mean_, (scx or sc).scale_, cols=cols_)
htgt = lambda c_: c_["truth"]["hail"]
hany = lambda c_: c_["S"]["I"]["hail"] > 0.02          # стресс-тесты: истина - любые удары града (в т. ч. слабые)
print("град, нормальная интенсивность (тест):"); display(pd.DataFrame([eval_events(mkh, test, htgt, "hail", pre=30, post=30)]).round(2))

### Стресс-тесты града
Обучение было на обычной интенсивности (I = 0,3-1). Проверяем (а) **слабый град** (I от 0,05); (б) **сильную турбулентность без града** (ветер 14 м/с: сценарии `clear` и `dust`); (в) **потери сообщений** (БВС-1 и БВС-7 по 30 %).

In [ ]:
def set_hail(I):
    def f(cfg):
        for e in cfg.events:
            if e.kind == "hail": e.I = I
    return f
rows = []
for I in (0.05, 0.1, 0.2, 0.4, 0.8):
    ch = build_cache3(["hail"], range(800, 808), modify=set_hail(I)); r = eval_events(mkh, ch, hany, "hail", pre=30, post=30); rows.append(dict(условие=f"град I={I}", **r))
windy = lambda cfg: setattr(cfg, "wind0", 14.0)
ch = build_cache3(["clear", "dust", "rain"], range(850, 858), modify=windy); r = eval_events(mkh, ch, htgt, "hail", pre=30, post=30); rows.append(dict(условие="ветер 14 м/с без града (clear, dust, rain)", **r))
hard = lambda cfg: (set_hail(0.1)(cfg), setattr(cfg, "wind0", 14.0))
ch = build_cache3(["hail"], range(850, 858), modify=hard); r = eval_events(mkh, ch, hany, "hail", pre=30, post=30); rows.append(dict(условие="слабый град I=0.1 + ветер 14 м/с", **r))
ch = build_cache3(["hail"], range(900, 908), net={"BVS1": (0.06, 0.4, 0.3), "BVS7": (0.25, 1.8, 0.3)}); r = eval_events(mkh, ch, hany, "hail", pre=30, post=30); rows.append(dict(условие="град, потери БВС-1 и БВС-7 по 30 %", **r))
stress = pd.DataFrame(rows).set_index("условие").round(2); display(stress)
# вклад источников при трудных условиях (слабый град + сильный ветер); добавлен прогноз БЕЗ индекса турбулентности (его связь с градом в генераторе - артефакт)
ch = build_cache3(["hail"], range(850, 858), modify=hard) + build_cache3(["rain", "dust", "clear"], range(870, 876), modify=windy); rows = []   # + негативы с осадками и ветром
sets2 = dict(sets); sets2["прогноз без turbulence_index"] = [4]
for name, cols_ in sets2.items():
    sc_ = StandardScaler().fit(Xtr[:, cols_]); lr_ = LogisticRegression(max_iter=500, class_weight="balanced").fit(sc_.transform(Xtr[:, cols_]), ytr)
    r = eval_events(lambda: mkh(cols_, sc_, lr_), ch, hany, "hail", pre=30, post=30); rows.append(dict(набор=name, recall=r["recall"], ложных_в_час=r["ложных_в_час"]))
display(pd.DataFrame(rows).set_index("набор").round(2))

## C. Перегрев от пыли: `cpu_temp`
Ожидаемая `cpu_temp = c₀ + c₁·(T − 20)` по полётам без пыли (T из БВС-5). Остаток сглаживается (EWMA 21 с, период БВС-7 ≈ 7 с), CUSUM вверх; отдельно - немедленный алерт по статусу ESC `warn` или коду `0x3105`. **Допущение:** нагрев процессора пылью (+14 °C при I = 1) и ESC-предупреждение - наши, физически правдоподобные, но не измеренные.

In [ ]:
dtgt = lambda c_: c_["truth"]["dust"]
r = eval_events(lambda: DustThermalDetector(cpu_model), test, dtgt, "dust_thermal", pre=0, post=60); display(pd.DataFrame([r]).round(2))
cd = next(c_ for c_ in test if c_["kind"] == "dust"); d = DustThermalDetector(cpu_model); d.reset(); ts, rs, al = [], [], []
for m in cd["msgs"]:
    if m["stream"] in d.streams:
        a = d.on_message(m)
        if m["stream"] == "BVS7" and m["p"]["system_health"]: ts.append(m["t_gen"]); rs.append(d.res)
        al += [m["t_gen"] for _ in a]
fig, ax = plt.subplots(figsize=(9, 3.4)); ax.plot(ts, rs, label="остаток cpu_temp (EWMA)"); ax.vlines(al, 0, 3, color="r", label="алерт")
for (s0, s1) in segments(cd["truth"]["dust"]): ax.axvspan(s0, s1, color="orange", alpha=.15, label="пыль (истина)")
ax.legend(fontsize=8); ax.set_xlabel("t, с"); ax.set_ylabel("°C"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Время обработки
Замер `perf_counter_ns` на сообщение каждого потока, который читает детектор. БВС-1 (≈ 8 Гц) - горячий путь для града.

In [ ]:
def timing(make, cache, n_scen=4):
    ts = []
    for c_ in cache[:n_scen]:
        d = make(); d.reset()
        for m in c_["msgs"]:
            if m["stream"] in d.streams:
                t1 = time.perf_counter_ns(); d.on_message(m); ts.append((time.perf_counter_ns() - t1) / 1e3)
    a = np.array(ts); return dict(сообщений=len(a), p50_мкс=np.percentile(a, 50), p99_мкс=np.percentile(a, 99), max_мкс=a.max())
tm = pd.DataFrame([dict(детектор="PowerResidual (БВС-1, БВС-5)", **timing(mk, test)), dict(детектор="Hail (БВС-1, БВС-5, БВС-7, НСУ-3)", **timing(mkh, test)),
                   dict(детектор="DustThermal (БВС-5, БВС-7)", **timing(lambda: DustThermalDetector(cpu_model), test))]).set_index("детектор").round(1)
tm["p99_мс при замедлении ×50"] = (tm.p99_мкс * 50 / 1000).round(2); display(tm)

## Выходной интерфейс
Алерты оформляются по полям БВС-3 (`alert_type` для погоды в ТЗ отсутствует - согласовать). К алерту добавляются количественные оценки: прибавка мощности и оценка изменения времени полёта (`estimated_flight_time` БВС-5 пересчитывается с учётом δ̂), «доза» града.

In [ ]:
def load_alert(t, delta_hat, T, uav_id="UAV-001"):
    icing = T < 1.0 and delta_hat >= 0.10
    return {"alert_id": f"pwr-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": 4 if icing else 3 if delta_hat >= 0.10 else 2,
            "description": "LOAD_INCREASE_" + ("ICING_RISK" if icing else "WET_OR_SNOW"), "sensor_source": "BVS5:power+BVS1:speed", "confidence": round(min(1.0, delta_hat / 0.1), 2),
            "load_increase_pct": round(100 * delta_hat, 1), "flight_time_change_pct": round(-100 * delta_hat / (1 + delta_hat), 1), "timestamp": int(t * 1000),
            "recommended_action": "plan_early_return; re-estimate_energy_budget(BVS10:route_replan)" + ("; leave_icing_layer" if icing else "")}
def hail_alert(t, p, n_events, uav_id="UAV-001"):
    return {"alert_id": f"hail-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": 4, "description": "HAIL_DETECTED", "sensor_source": "BVS1+BVS7+NSU3",
            "confidence": round(float(p), 2), "impact_events_15s": int(n_events), "timestamp": int(t * 1000), "recommended_action": "emergency_land_or_RTH; log_impacts; request_zone_check"}
display(load_alert(250.0, 0.07, -2.0)); display(hail_alert(180.0, 0.93, 11))

## 8. Выводы и ограничения
**A. Налипание снега и намокание (остаток мощности).**
- Модель ожидаемой мощности по скорости и *сглаженным* порывам даёт остаток с σ ≈ 3,2 %. Без сглаживания порывов (шум измерения) порывы при граде и пыли давали ложные тревоги; без прогрева 25 с первая секунда полёта давала тревогу на каждом старте.
- Детектор «сглаженная прибавка δ̂ + Gate» (τ = 30 с, порог 2 %): на независимых сценариях **полнота 1,0, 1 алерт на событие, медианная задержка ≈ 15 с** после того, как прибавка превысила 2,5 %, **0,72 ложных тревоги в час** (сосредоточены в сценариях с дождём 3,0/ч, градом 3,75/ч и пылью 1,5/ч, где порывы и нагрузка меняются). CUSUM: полнота 0,91 при той же доле ложных тревог.
- Оценка уровня прибавки: средняя ошибка δ̂ около 0,9 процентного пункта при δ > 2 %.
- **Граница обнаружения ≈ 2-3 % прибавки мощности:** при слабом снеге (I = 0,15, максимум 1,8 %) событий нет, а срабатывания (6/ч) расходятся с истиной; при I = 0,3 (3,6 %) полнота 0,5; при I ≥ 0,6 (≥ 7 %) полнота 1,0, задержка 20-25 с. Намокание от дождя (+3 % × I) почти всегда ниже границы: **дождь по мощности обнаружить нельзя**, снег и обледенение - можно.
- Прибавка нагрузки позволяет пересчитать запас времени полёта (алерт содержит `flight_time_change_pct`).

**B. Град.**
- Все три источника по отдельности разделяют град в симуляторе очень хорошо (AUROC: скачки attitude 0,986; ошибки IMU 0,991; прогноз НСУ-3 0,997; вместе 1,0). Градовый детектор на тесте: **полнота 1,0, задержка ≈ 2 с, ложных тревог нет**.
- Стресс-тесты: слабый град (I = 0,05) - полнота 1,0, задержка ≈ 7 с; слабый град при ветре 14 м/с - полнота 1,0, задержка ≈ 5 с; потери БВС-1 и БВС-7 по 30 % - полнота 1,0, задержка ≈ 2 с; ветер 14 м/с без града - ложных нет.
- **Оговорка:** разделимость завышена генератором (удары - импульсы 3-8° и коды IMU, град всегда сопровождается дождём, а `turbulence_index` в НСУ-3 содержит вклад града). Сравнение источников на выборке с дождём, пылью и ясной погодой при сильном ветре: прогноз НСУ-3 один даёт 6 ложных в час, без `turbulence_index` - 1,4; скачки attitude - 0,5; ошибки IMU - 0, но в реальном потоке код `0x2101` и его интерпретация неизвестны. Лучше всего работает комбинация (0 ложных). **Для реальной оценки нужны лётные логи с градом.**
- Защита - только постфактум (реакция через ≈ 2-7 с после первых ударов): аварийная посадка или возврат, но не предотвращение.

**C. Перегрев от пыли (`cpu_temp`).** Полнота 1,0, задержка ≈ 5 с, ложных нет. Это зависит от допущения о нагреве процессора пылью (+14 °C при I = 1), которое нужно проверить на реальных данных; рост температуры может иметь и другие причины (нагрузка, жаркая погода учтена линейной моделью по T).

**Время.** Остаток мощности: p99 ≈ 3 мкс на сообщение; перегрев: p99 ≈ 2 мкс; град (горячий путь БВС-1 ≈ 8 Гц, медиана по окну 80 отсчётов): p50 ≈ 48 мкс, p99 ≈ 90 мкс, максимум ≈ 0,7 мс (при замедлении ×50 p99 ≈ 4,5 мс, у границы быстрого контура ≲ 5 мс). Для борта медиану окна желательно заменить на более дешёвую оценку (EWMA |x|); это не проверялось.

**Ограничения.** Все физические связи (прибавка от снега до 25 %, нагрев пылью, удары града) - допущения генератора; нет влияния высоты, заряда батареи и массы груза на мощность (в реальности эти факторы дают дополнительный остаток и требуют калибровки по конкретному аппарату); поток БВС-7 редкий (0,14 Гц), поэтому перегрев и ESC видны с задержкой 5-10 с.

**Дальше:** ноутбук 04 (слияние видимости: `visibility_m` НСУ-3, дальности и уверенность БВС-4, метео БВС-5 → оценка видимости и доверия сенсорам), затем интеграция в индекс пригодности условий с итоговым замером задержек и ложных тревог на смешанных сценариях.